# I will use the following notebook to demonstrate different steps in preprocessing

## These steps will include:

### 1) Slice timing correction
### 2) Motion correction
### 3) Coregistration
### 4) Spatial and temporal filtering

In [2]:
%pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 117.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 83.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 81.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from glob import glob

# Import new things that we'll need
import pandas as pd
import numpy as np
import nipype.interfaces.afni as afni
import nipype.interfaces.fsl as fsl
import nipype.interfaces.freesurfer as fs
from nipype.interfaces.utility import Function
import nibabel as nb
import json
import nipype.interfaces.io as nio
import nipype.pipeline.engine as pe 
import matplotlib.pyplot as plt

%matplotlib inline

print("All Python libraries imported")

All Python libraries imported


### I next want to get a list of all of my functional files

In [21]:
sid = ['021']
base_dir = '/home/umcca001/Mattfeld_PSB6351'
work_dir = '/home/umcca001/psb6351_preproc_work'
func_dir = os.path.join(base_dir, f'dset/sub-{sid[0]}/func')
fmap_dir = os.path.join(base_dir, f'dset/sub-{sid[0]}/fmap')
fs_dir = os.path.join(base_dir, 'derivatives', 'freesurfer')

# Get a list of my study task json and nifti converted files
func_json = sorted(glob(func_dir + '/*.json'))
func_files = sorted(glob(func_dir + '/*.nii.gz'))
fmap_files = sorted(glob(fmap_dir + '/*func*.nii.gz'))

In [22]:
print(fs_dir)
print(os.path.isdir(fs_dir))

/home/umcca001/Mattfeld_PSB6351/derivatives/freesurfer
False


In [8]:
print(glob(fmap_dir + '/*'))

['/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-dMRI_dir-AP_epi.json', '/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-fMRI_dir-AP_epi.nii.gz', '/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-fMRI_dir-PA_epi.nii.gz', '/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-fMRI_dir-PA_epi.json', '/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-fMRI_dir-AP_epi.json', '/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-dMRI_dir-AP_epi.nii.gz', '/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-dMRI_dir-PA_epi.json', '/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-dMRI_dir-PA_epi.nii.gz']


In [9]:
fmap_files = sorted(glob(fmap_dir + '/*acq-fMRI*.nii.gz'))

In [10]:
fmap_files[0]

'/home/umcca001/Mattfeld_PSB6351/dset/sub-021/fmap/sub-021_acq-fMRI_dir-AP_epi.nii.gz'

### Next I want to build and run function to perform slice timing correction. I'm going to have to extract some important information from the .json files like the multiband slicetiming.

In [23]:
# Here I am building a function that eliminates the
# mapnode directory structure and assists in saving
# all of the outputs into a single directory
def get_subs(func_files):
    '''Produces Name Substitutions for Each Contrast'''
    subs = []
    for curr_run in range(len(func_files)):
        subs.append(('_mapflow/_volreg%d/' %curr_run, ''))
    return subs

# Here I am building a function that takes in a
# text file that includes the number of outliers
# at each volume and then finds which volume (e.g., index)
# has the minimum number of outliers (e.g., min)
# searching over the first 201 volumes
# If the index function returns a list because there were
# multiple volumes with the same outlier count, pick the first one
def best_vol(outlier_count):
    import numpy as np
    outlier_count = np.loadtxt(outlier_count)
    best_vol_num = int(np.argmin(outlier_count[:200]))
    if isinstance(best_vol_num, list):
        best_vol_num = best_vol_num[0]
    return best_vol_num

# Here I am creating a list of lists containing the slice timing for each study run
slice_timing_list = []
for curr_json in func_json:
    curr_json_data = open(curr_json)
    curr_func_metadata = json.load(curr_json_data)
    slice_timing_list.append(curr_func_metadata['SliceTiming'])

# Here I am establishing a nipype work flow that I will eventually execute
psb6351_wf = pe.Workflow(name='psb6351_wf')
psb6351_wf.base_dir = work_dir + f'/psb6351workdir/sub-{sid[0]}'
psb6351_wf.config['execution']['use_relative_paths'] = True

# Create a Function node to substitute names of files created during pipeline
getsubs = pe.Node(Function(input_names=['func_files'],
                           output_names=['subs'],
                           function=get_subs),
                  name='getsubs')
getsubs.inputs.func_files = func_files

# Here I am inputing just the first run functional data
# I want to use afni's 3dToutcount to find the number of
# outliers at each volume. I will use this information to
# later select the earliest volume with the least number of outliers
# to serve as the base for the motion correction
id_outliers = pe.Node(afni.OutlierCount(),
                      name = 'id_outliers')
# USE AFNI'S OUTLIER COUNT TO GET NUMBER OF OUTLIERS PER RUN

id_outliers.inputs.automask = True
id_outliers.inputs.legendre = True
id_outliers.inputs.polort = 4
id_outliers.inputs.out_file = 'outlier_file'
id_outliers.inputs.in_file = func_files[0]

#ATM ONLY: Add an unwarping mapnode here using the field maps
#EXTRA CREDIT FIGURE OUT HOW TO ADD A NODE FOR AFNI'S QWARP
#TO UTILIZE THE DISTORTION CORRECTION FIELD MAPS

#NOW CREATE ANOTHER NODE TO APPLY THE WARPS

# Create a Function node to identify the best volume based
# on the number of outliers at each volume. I'm searching
# for the index in the first 201 volumes that has the
# minimum number of outliers and will use the min() function
# I will use the index function to get the best vol.

getbestvol = pe.Node(Function(input_names=['outlier_count'],
                              output_names=['best_vol_num'],
                              function=best_vol),
                     name='getbestvol')

psb6351_wf.connect(id_outliers, 'out_file', getbestvol, 'outlier_count')

# Extract the earliest volume with the
# the fewest outliers of the first run as the reference

# USE FSL EXTRACT ROI TO SELECT THE VOLUME FOR MOTION CORRECTION

extractref = pe.Node(fsl.ExtractROI(t_size=1),
                     name='extractref')

extractref.inputs.in_file = func_files[0]

psb6351_wf.connect(getbestvol, 'best_vol_num', extractref, 't_min')

# Below is the command that runs AFNI's 3dvolreg command.
# this is the node that performs the motion correction
# I'm iterating over the functional files which I am passing
# functional data from the slice timing correction node before
# I'm using the earliest volume with the least number of outliers
# during the first run as the base file to register to.

# USE AFNIs 3dvolreg to perform the motion correction.

volreg = pe.MapNode(afni.Volreg(),
                    iterfield=['in_file'],
                    name='volreg')

volreg.inputs.outputtype = 'NIFTI_GZ'
volreg.inputs.zpad = 4
volreg.inputs.in_file = func_files

psb6351_wf.connect(extractref, 'roi_file', volreg, 'basefile')

# Below is the command that runs AFNI's 3dTshift command
# this is the node that performs the slice timing correction
# I input the study func files as a list and the slice timing
# as a list of lists. I'm using a MapNode to iterate over the two.
# this should allow me to parallelize this on the HPC

# USE AFNIs 3dTSHIFT TO DO THE SLICE TIMING CORRECTION

tshifter = pe.MapNode(afni.TShift(),
                      iterfield=['in_file', 'slice_timing'],
                      name='tshifter')

tshifter.inputs.outputtype = 'NIFTI_GZ'
tshifter.inputs.slice_timing = slice_timing_list

psb6351_wf.connect(volreg, 'out_file', tshifter, 'in_file')

# Calculate the transformation matrix from EPI space to FreeSurfer space
# using the BBRegister command

# USE FREESURFERs BBREGISTER TO DO THE COREGISTRATION

# BBRegister is temporarily skipped because the FreeSurfer
# directory for subject 021 does not exist at the specified path

# bbregister = pe.Node(fs.BBRegister(),
#                      name='bbregister')

# bbregister.inputs.subject_id = 'sub-021'
# bbregister.inputs.init = 'fsl'
# bbregister.inputs.contrast_type = 't2'
# bbregister.inputs.subjects_dir = fs_dir

# Below is the node that collects all the data and saves
# the outputs that I am interested in. Here in this node
# I use the substitutions input combined with the earlier
# function to get rid of nesting

# SAVE EVERYTHING WITH DATASINK

datasink = pe.Node(nio.DataSink(), name="datasink")

datasink.inputs.base_directory = os.path.join(
    base_dir, f'derivatives/sub-{sid[0]}'
)

psb6351_wf.connect(extractref, 'roi_file', datasink, 'reference.@ref')
psb6351_wf.connect(volreg, 'out_file', datasink, 'motion.@corrfile')
psb6351_wf.connect(volreg, 'oned_file', datasink, 'motion.@parameters')
psb6351_wf.connect(tshifter, 'out_file', datasink, 'sltime_corr')
psb6351_wf.connect(getsubs, 'subs', datasink, 'substitutions')

# The following two lines set a work directory outside of my
# local git repo and runs the workflow

psb6351_wf.run(plugin='SLURM',
               plugin_args={'sbatch_args': ('--partition highmem1-sapphirerapids --qos classroom --account acc_psb6351'),
                            'overwrite':True})

261012-15:28:58,305 nipype.interface WARNING:
	 FSLOUTPUTTYPE environment variable is not set. Setting FSLOUTPUTTYPE=NIFTI
261012-15:28:58,315 nipype.workflow INFO:
	 Workflow psb6351_wf settings: ['check', 'execution', 'logging', 'monitoring']
261012-15:28:58,406 nipype.workflow INFO:
	 Running in parallel.
261012-15:28:58,446 nipype.workflow INFO:
	 Pending[0] Submitting[2] jobs Slots[inf]
261012-15:28:58,446 nipype.workflow INFO:
	 Submitting: psb6351_wf.id_outliers ID: 0


IndexError: list index out of range

### Now I will load and plot the motion files

In [18]:
motion_dir = os.path.join(base_dir, f'derivatives/sub-{sid[0]}/motion')
study_motion_files = sorted(glob(motion_dir + '/*.1D'))

for curr_mot_file in study_motion_files:

for curr_mot_file in study_motion_files:

    motion_data = np.loadtxt(curr_mot_file)
    plt.plot(motion_data)

    plt.show()
    
SHOULD LOOK LIKE OUTPUT BELOW IN THE END...NOTE THE SCALE ON Y-AXIS

IndentationError: expected an indented block (725674506.py, line 6)